# NLP Project — Medical Chatbot
## Step 3: Prepare Sequences (Tokenize → Pad → Save Tensors)
**Input:** `data/train.txt`, `data/val.txt`, `data/test.txt` + `tokenizer/medical_bpe.model`  
**Output:** `data/train_tokens.pt`, `data/train_masks.pt` (and same for val, test)

### Cell 1 — Load Config and Tokenizer

In [1]:
import json
import torch
import sentencepiece as spm

# Load config saved in Step 2
with open('../tokenizer/config.json', 'r') as f:
    config = json.load(f)

MAX_SEQ_LEN = config['max_seq_len']   # 512
PAD_ID      = config['pad_id']        # 0

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(config['model_path'])

print(f"Config loaded ✓")
print(f"MAX_SEQ_LEN : {MAX_SEQ_LEN}")
print(f"PAD_ID      : {PAD_ID}")
print(f"Vocab size  : {sp.get_piece_size()}")

Config loaded ✓
MAX_SEQ_LEN : 512
PAD_ID      : 0
Vocab size  : 8000


### Cell 2 — Core Function: Tokenize + Pad + Mask

For each sequence:
1. Encode text → token IDs
2. Truncate if longer than 512
3. Pad with 0s if shorter than 512
4. Build attention mask: 1 for real tokens, 0 for padding

In [2]:
def tokenize_and_pad(sequences, sp, max_len, pad_id):
    """
    Takes a list of text sequences.
    Returns:
        tokens_tensor : LongTensor of shape (N, max_len)
        masks_tensor  : LongTensor of shape (N, max_len)
    """
    all_tokens = []
    all_masks  = []

    for seq in sequences:
        # Step 1: Encode
        ids = sp.encode(seq, out_type=int)

        # Step 2: Truncate if too long
        ids = ids[:max_len]

        # Step 3: Build mask BEFORE padding (1 = real token)
        mask = [1] * len(ids)

        # Step 4: Pad to max_len
        pad_len = max_len - len(ids)
        ids  = ids  + [pad_id] * pad_len
        mask = mask + [0]      * pad_len

        all_tokens.append(ids)
        all_masks.append(mask)

    tokens_tensor = torch.tensor(all_tokens, dtype=torch.long)
    masks_tensor  = torch.tensor(all_masks,  dtype=torch.long)

    return tokens_tensor, masks_tensor

print("tokenize_and_pad() ready ✓")

tokenize_and_pad() ready ✓


### Cell 3 — Quick Sanity Check on 3 Samples

In [3]:
# Test with 3 dummy sequences before running on full data
test_seqs = [
    "<patient> I have fever and headache since 2 days. <doctor> Take rest and paracetamol. <eos>",
    "<patient> Short. <doctor> Ok. <eos>",
    "<patient> " + "word " * 300 + "<doctor> long response <eos>"  # intentionally long
]

tok, msk = tokenize_and_pad(test_seqs, sp, MAX_SEQ_LEN, PAD_ID)

print(f"Tokens shape : {tok.shape}")
print(f"Masks  shape : {msk.shape}")
print()

for i, seq in enumerate(test_seqs):
    real_len = msk[i].sum().item()
    print(f"Seq {i+1}: real tokens = {real_len}, padded = {MAX_SEQ_LEN - real_len}")
    print(f"  First 10 token ids : {tok[i][:10].tolist()}")
    print(f"  First 10 mask vals : {msk[i][:10].tolist()}")
    print(f"  Last  10 token ids : {tok[i][-10:].tolist()}")
    print(f"  Last  10 mask vals : {msk[i][-10:].tolist()}")
    print()

Tokens shape : torch.Size([3, 512])
Masks  shape : torch.Size([3, 512])

Seq 1: real tokens = 20, padded = 492
  First 10 token ids : [7831, 5, 36, 81, 419, 32, 1243, 586, 205, 310]
  First 10 mask vals : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  Last  10 token ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Last  10 mask vals : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Seq 2: real tokens = 11, padded = 501
  First 10 token ids : [7831, 5, 5789, 356, 7850, 7831, 6, 5941, 7850, 7831]
  First 10 mask vals : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  Last  10 token ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Last  10 mask vals : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Seq 3: real tokens = 512, padded = 0
  First 10 token ids : [7831, 5, 239, 7841, 239, 7841, 239, 7841, 239, 7841]
  First 10 mask vals : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  Last  10 token ids : [239, 7841, 239, 7841, 239, 7841, 239, 7841, 239, 7841]
  Last  10 mask vals : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]



### Cell 4 — Process Train Split

In [4]:
print("Loading train.txt...")
with open('../data/train.txt', 'r', encoding='utf-8') as f:
    train_seqs = [line.strip() for line in f if line.strip()]

print(f"Sequences loaded : {len(train_seqs):,}")
print("Tokenizing and padding...")

train_tokens, train_masks = tokenize_and_pad(train_seqs, sp, MAX_SEQ_LEN, PAD_ID)

print(f"\ntrain_tokens shape : {train_tokens.shape}")
print(f"train_masks  shape : {train_masks.shape}")
print(f"Memory (tokens)    : {train_tokens.element_size() * train_tokens.nelement() / 1024**2:.1f} MB")

torch.save(train_tokens, '../data/train_tokens.pt')
torch.save(train_masks,  '../data/train_masks.pt')
print("\nSaved: data/train_tokens.pt")
print("Saved: data/train_masks.pt  ✓")

Loading train.txt...
Sequences loaded : 81,049
Tokenizing and padding...

train_tokens shape : torch.Size([81049, 512])
train_masks  shape : torch.Size([81049, 512])
Memory (tokens)    : 316.6 MB

Saved: data/train_tokens.pt
Saved: data/train_masks.pt  ✓


### Cell 5 — Process Validation Split

In [5]:
print("Loading val.txt...")
with open('../data/val.txt', 'r', encoding='utf-8') as f:
    val_seqs = [line.strip() for line in f if line.strip()]

print(f"Sequences loaded : {len(val_seqs):,}")
print("Tokenizing and padding...")

val_tokens, val_masks = tokenize_and_pad(val_seqs, sp, MAX_SEQ_LEN, PAD_ID)

print(f"\nval_tokens shape : {val_tokens.shape}")
print(f"val_masks  shape : {val_masks.shape}")

torch.save(val_tokens, '../data/val_tokens.pt')
torch.save(val_masks,  '../data/val_masks.pt')
print("\nSaved: data/val_tokens.pt")
print("Saved: data/val_masks.pt  ✓")

Loading val.txt...
Sequences loaded : 10,120
Tokenizing and padding...

val_tokens shape : torch.Size([10120, 512])
val_masks  shape : torch.Size([10120, 512])

Saved: data/val_tokens.pt
Saved: data/val_masks.pt  ✓


### Cell 6 — Process Test Split

In [6]:
print("Loading test.txt...")
with open('../data/test.txt', 'r', encoding='utf-8') as f:
    test_seqs = [line.strip() for line in f if line.strip()]

print(f"Sequences loaded : {len(test_seqs):,}")
print("Tokenizing and padding...")

test_tokens, test_masks = tokenize_and_pad(test_seqs, sp, MAX_SEQ_LEN, PAD_ID)

print(f"\ntest_tokens shape : {test_tokens.shape}")
print(f"test_masks  shape : {test_masks.shape}")

torch.save(test_tokens, '../data/test_tokens.pt')
torch.save(test_masks,  '../data/test_masks.pt')
print("\nSaved: data/test_tokens.pt")
print("Saved: data/test_masks.pt  ✓")

Loading test.txt...
Sequences loaded : 10,130
Tokenizing and padding...

test_tokens shape : torch.Size([10130, 512])
test_masks  shape : torch.Size([10130, 512])

Saved: data/test_tokens.pt
Saved: data/test_masks.pt  ✓


### Cell 7 — Verify All Saved Files

In [7]:
import os

files = [
    '../data/train_tokens.pt', '../data/train_masks.pt',
    '../data/val_tokens.pt',   '../data/val_masks.pt',
    '../data/test_tokens.pt',  '../data/test_masks.pt',
]

print("=== Verifying saved files ===")
print(f"{'File':<30} {'Shape':<20} {'Size (MB)':>10}")
print("-" * 62)

for fpath in files:
    t = torch.load(fpath, weights_only=True)
    size_mb = os.path.getsize(fpath) / 1024**2
    print(f"{fpath.split('/')[-1]:<30} {str(tuple(t.shape)):<20} {size_mb:>10.1f} MB")

print()
print("All files verified ✓")

=== Verifying saved files ===
File                           Shape                 Size (MB)
--------------------------------------------------------------
train_tokens.pt                (81049, 512)              316.6 MB
train_masks.pt                 (81049, 512)              316.6 MB
val_tokens.pt                  (10120, 512)               39.5 MB
val_masks.pt                   (10120, 512)               39.5 MB
test_tokens.pt                 (10130, 512)               39.6 MB
test_masks.pt                  (10130, 512)               39.6 MB

All files verified ✓


### Cell 8 — Final Check: Spot-Check One Sample End to End

In [8]:
# Reload from disk and decode sample 0 to confirm everything is correct
tokens = torch.load('../data/train_tokens.pt', weights_only=True)
masks  = torch.load('../data/train_masks.pt',  weights_only=True)

sample_idx = 0
sample_tokens = tokens[sample_idx].tolist()
sample_mask   = masks[sample_idx].tolist()

# Only decode the real (non-padded) tokens
real_len    = sum(sample_mask)
real_tokens = sample_tokens[:real_len]

decoded = sp.decode(real_tokens)

print(f"Sample index     : {sample_idx}")
print(f"Real token count : {real_len}")
print(f"Pad token count  : {MAX_SEQ_LEN - real_len}")
print()
print("Decoded text:")
print(decoded)

# Confirm it matches original
with open('../data/train.txt', 'r', encoding='utf-8') as f:
    original = [line.strip() for line in f if line.strip()][sample_idx]

print()
print(f"Matches original : {decoded.strip() == original.strip()}")

Sample index     : 0
Real token count : 152
Pad token count  : 360

Decoded text:
<patient> Can differentiation in the platelets and ITP be cured? My daughter has low blood platelets she just got it tested and tare at 80000 she has had ts in the past where she was diagnosed a year ago with ITP can ts be cured? She was put on predizone then her platelets went up to 150000 however thave been going down and now tare at 80000 wch I believe tshould be between 150000 to 300000 <doctor> if she is otherwise fine with no bleeds, platelets of 80000 is okay. It can be observed over time. if it falls below 40000, we may increase the dose of prednisolone or add another drug.often ITP gets better over time and may be cured.Hope ts helps. <eos>

Matches original : True


### Cell 9 — Summary Stats

In [9]:
tokens = torch.load('../data/train_tokens.pt', weights_only=True)
masks  = torch.load('../data/train_masks.pt',  weights_only=True)

real_lengths = masks.sum(dim=1).float()

print("=== Step 3 Complete — Final Stats ===")
print()
print(f"Train sequences     : {tokens.shape[0]:,}")
print(f"Sequence length     : {tokens.shape[1]}  (all padded to this)")
print()
print(f"Avg real tokens     : {real_lengths.mean().item():.1f}")
print(f"Min real tokens     : {real_lengths.min().item():.0f}")
print(f"Max real tokens     : {real_lengths.max().item():.0f}")
print(f"% sequences at 512  : {(real_lengths == 512).float().mean().item()*100:.2f}%  (truncated)")
print()
print("Files ready for Step 4 (Model Building):")
print("  data/train_tokens.pt  data/train_masks.pt")
print("  data/val_tokens.pt    data/val_masks.pt")
print("  data/test_tokens.pt   data/test_masks.pt")
print("  tokenizer/config.json")

=== Step 3 Complete — Final Stats ===

Train sequences     : 81,049
Sequence length     : 512  (all padded to this)

Avg real tokens     : 212.9
Min real tokens     : 42
Max real tokens     : 512
% sequences at 512  : 0.36%  (truncated)

Files ready for Step 4 (Model Building):
  data/train_tokens.pt  data/train_masks.pt
  data/val_tokens.pt    data/val_masks.pt
  data/test_tokens.pt   data/test_masks.pt
  tokenizer/config.json
